### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# %%bash
# cd ./
# rm *.keras

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from PIL import ImageOps
from tensorflow.keras import regularizers
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
train_ds, train_metadata = tfds.load('Food101', split='train', shuffle_files=True, as_supervised=True, batch_size=128, with_info=True)
val_ds, val_metadata = tfds.load('Food101', split='validation', shuffle_files=True, as_supervised=True, batch_size=128, with_info=True)

In [ ]:
print(train_metadata.features["label"].num_classes)
print(train_metadata.features["label"].names)

In [ ]:
sample_data = list(train_ds.take(1))

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(8):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(train_metadata.features["label"].int2str(labels[i].numpy().astype("uint8")))
        plt.axis("off")

In [ ]:
from tqdm import tqdm

data_info = {
    class_name: 0
    for class_name in train_metadata.features["label"].names
}
for _, label in tqdm(train_ds):
  for value in label:
    class_id = int(value)
    class_name = train_metadata.features["label"].int2str(class_id)
    data_info[class_name] += 1

In [ ]:
meals = list(data_info.keys())
number_of_images = list(data_info.values())

fig = plt.figure(figsize = (20, 10))

# creating the bar plot
plt.bar(meals, number_of_images, color ='maroon',
        width = 0.6)

# Rotation of the bars names
plt.xticks(range(len(meals)), meals, rotation='vertical')

plt.xlabel("Meals")
plt.ylabel("Number of images")
plt.title("Number of images for each meal")
plt.show()

In [ ]:
IMAGE_SIZE = (256, 256)
L1_LAMBDA = 1e-4
L2_LAMBDA = 1e-4

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(8):
        ax = plt.subplot(3, 3, i + 1)
        img = layers.RandomBrightness(factor=0.3, value_range=(0, 255))(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(train_metadata.features["label"].int2str(labels[i].numpy().astype("uint8")))
        plt.axis("off")

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(8):
        ax = plt.subplot(3, 3, i + 1)
        img = layers.RandomFlip("horizontal")(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(train_metadata.features["label"].int2str(labels[i].numpy().astype("uint8")))
        plt.axis("off")

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(8):
        ax = plt.subplot(3, 3, i + 1)
        img = layers.RandomRotation(0.3)(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(train_metadata.features["label"].int2str(labels[i].numpy().astype("uint8")))
        plt.axis("off")

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomBrightness(factor=0.3, value_range=(0, 255)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.3),
    ]
)

resize_layer = layers.Resizing(*IMAGE_SIZE)

# Apply `data_augmentation` to the training images.
train_ds = train_ds.map(
    lambda img, label: (resize_layer(img), label),
    num_parallel_calls=tf.data.AUTOTUNE,
)
val_ds = val_ds.map(
    lambda img, label: (resize_layer(img), label),
    num_parallel_calls=tf.data.AUTOTUNE,
)

train_ds = train_ds.map(
    lambda img, label: (data_augmentation(img), label),
    num_parallel_calls=tf.data.AUTOTUNE,
)

# Prefetching samples in GPU memory helps maximize GPU utilization.
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Entry block
    x = layers.Rescaling(1.0 / 255)(inputs)
    backbone = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet"
    )
    x = backbone(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu",
                      kernel_regularizer=regularizers.L1(L1_LAMBDA),
                      activity_regularizer=regularizers.L2(L2_LAMBDA))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)
model = make_model(input_shape=IMAGE_SIZE + (3,), num_classes=101)

In [ ]:
model = tf.keras.models.load_model("./")

In [ ]:
epochs = 7

callbacks = [
    keras.callbacks.ModelCheckpoint("./"),
]
# model.compile(
#     optimizer=keras.optimizers.Adam(1e-3),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"],
# )
history = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds
)

In [ ]:
history = {
  "loss": [],
  "accuracy": [],
  "val_loss": [],
  "val_accuracy": [],
}

for i in range(1, 6 + 1):
  file_path = f'./'
  with open(file_path) as f:
    data = json.loads(f.read())
  for key in history:
    history[key].extend(data[key])

In [ ]:
from matplotlib import pyplot as plt
plt.plot(history['accuracy'])
plt.plot(history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history['loss'])
plt.plot(history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
history.history

In [ ]:
import json

json.dump(history, open( "./", 'w' ))

# Test

In [ ]:
loaded_model = tf.keras.models.load_model("./")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(21, 21))
for images, labels in val_ds.take(1):
    for i in range(6*6):
        ax = plt.subplot(6, 6, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        img_array = tf.expand_dims(images[i].numpy().astype("uint8"), 0)  # Create batch axis
        preds = loaded_model.predict(img_array)
        class_id = np.argmax(preds[0])
        class_name = train_metadata.features["label"].int2str(class_id)
        plt.title(class_name)
        plt.axis("off")